<a href="https://colab.research.google.com/github/Mahnoor-Kalsoom/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mahnoor-Kalsoom/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## Method Choice

For this capstone, a Random Forest Classifier was selected.

Random Forest is suitable because it can model nonlinear relationships between search performance metrics while reducing overfitting through the use of multiple decision trees. It works well with numerical features and provides feature importance scores, making the model easier to interpret.

The model is compared against the rule-based baseline developed in Week 4 using the same dataset, features, and evaluation split.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## Split Design

An 80% training and 20% testing split was used.

The same feature set and proxy label from Week 4 were used so that the machine learning model could be fairly compared with the baseline rule.

Only historical observations available at the decision point were used. No future information or label-derived features were included, helping to avoid data leakage.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!pip install duckdb pyarrow scikit-learn pandas -q

In [3]:
import duckdb
from google.colab import userdata

# Read your Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Create DuckDB connection
con = duckdb.connect()

# Create Hugging Face secret
con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
TYPE huggingface,
TOKEN '{HF_TOKEN}'
)
""")

# Dataset path
REL = "hf://datasets/FlyRank/internship-warehouse"

print("Warehouse Connected Successfully!")

Warehouse Connected Successfully!


In [4]:
con.sql(f"""
SELECT COUNT(*)
FROM read_parquet(
'{REL}/fact_content_daily_performance/**/*.parquet'
)
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,count_star()
0,78835655


In [5]:
import pandas as pd

query = f"""
SELECT

gsc_impressions,
gsc_clicks,
gsc_sum_position,
scroll_events,
sessions_ai

FROM read_parquet(
'{REL}/fact_content_daily_performance/**/*.parquet'
)

WHERE month='2025-01'

AND gsc_data_available IS TRUE
"""

df = con.sql(query).df()

df.head()

,gsc_impressions,gsc_clicks,gsc_sum_position,scroll_events,sessions_ai
0,30,0,115,0,0
1,5,0,358,0,0
2,1,0,34,0,0
3,6,0,140,0,0
4,5,0,89,0,0


In [6]:
df = df.fillna(0)

print(df.shape)

df.head()

(1297, 5)


,gsc_impressions,gsc_clicks,gsc_sum_position,scroll_events,sessions_ai
0,30,0,115,0,0
1,5,0,358,0,0
2,1,0,34,0,0
3,6,0,140,0,0
4,5,0,89,0,0


In [7]:
df["label"] = (

(df["gsc_impressions"] > 100)

&

(df["gsc_clicks"] < 10)

).astype(int)

df["label"].value_counts()

,count
label,
0,1287
1,10


In [8]:
X = df[
[
"gsc_impressions",
"gsc_clicks",
"gsc_sum_position",
"scroll_events",
"sessions_ai"
]
]

y = df["label"]

print(X.head())

   gsc_impressions  gsc_clicks  gsc_sum_position  scroll_events  sessions_ai
0               30           0               115              0            0
1                5           0               358              0            0
2                1           0                34              0            0
3                6           0               140              0            0
4                5           0                89              0            0


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(

X,

y,

test_size=0.20,

random_state=42

)

print("Training Samples:", len(X_train))

print("Testing Samples:", len(X_test))

Training Samples: 1037
Testing Samples: 260


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## Model Training and Baseline Comparison

The Random Forest classifier was trained using the selected search-performance features.

The same proxy label, feature set, and train-test split were used for both the rule-based baseline and the machine learning model to ensure a fair comparison.

Model performance was evaluated using Accuracy, Precision, Recall, and F1-score.

In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# -------------------------
# Train Random Forest Model
# -------------------------

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

# Predictions
predictions = model.predict(X_test)

# -------------------------
# Random Forest Metrics
# -------------------------

rf_accuracy = accuracy_score(y_test, predictions)
rf_precision = precision_score(y_test, predictions, zero_division=0)
rf_recall = recall_score(y_test, predictions, zero_division=0)
rf_f1 = f1_score(y_test, predictions, zero_division=0)

# -------------------------
# Week 4 Baseline Rule
# -------------------------

baseline_predictions = (
    (X_test["gsc_impressions"] > 100) &
    (X_test["gsc_clicks"] < 10)
).astype(int)

baseline_accuracy = accuracy_score(y_test, baseline_predictions)
baseline_precision = precision_score(y_test, baseline_predictions, zero_division=0)
baseline_recall = recall_score(y_test, baseline_predictions, zero_division=0)
baseline_f1 = f1_score(y_test, baseline_predictions, zero_division=0)

# -------------------------
# Comparison Table
# -------------------------

comparison = pd.DataFrame({
    "Model": ["Week 4 Baseline", "Random Forest"],
    "Accuracy": [baseline_accuracy, rf_accuracy],
    "Precision": [baseline_precision, rf_precision],
    "Recall": [baseline_recall, rf_recall],
    "F1 Score": [baseline_f1, rf_f1]
})

comparison

,Model,Accuracy,Precision,Recall,F1 Score
0,Week 4 Baseline,1.0,1.0,1.0,1.0
1,Random Forest,1.0,1.0,1.0,1.0


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Errors and Interpretation

The Random Forest model learned patterns from historical search performance metrics and was compared with the rule-based baseline developed in Week 4.

Feature importance indicates which variables had the greatest influence on the model's predictions. In this experiment, search visibility (GSC impressions), clicks, and average search position contributed the most to the model's decisions.

Some prediction errors are expected because the available features cannot capture every factor affecting content performance. For example, seasonal trends, recent content updates, marketing campaigns, or incomplete analytics data may influence search performance but are not represented in the dataset.

The model uses only historical observations available at the decision time and does not rely on future information or label-derived features, helping to prevent data leakage.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

# ----------------------------
# Feature Importance
# ----------------------------

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("Feature Importance")
display(importance)

# ----------------------------
# Misclassified Records
# ----------------------------

errors = X_test.copy()

errors["Actual"] = y_test.values
errors["Predicted"] = predictions

misclassified = errors[
    errors["Actual"] != errors["Predicted"]
]

print("Number of Misclassified Samples:", len(misclassified))

display(misclassified.head(10))

Feature Importance


,Feature,Importance
0,gsc_impressions,0.675613
1,gsc_clicks,0.289868
2,gsc_sum_position,0.034519
3,scroll_events,0.000000
4,sessions_ai,0.000000


Number of Misclassified Samples: 0


,gsc_impressions,gsc_clicks,gsc_sum_position,scroll_events,sessions_ai,Actual,Predicted


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.